# fal-client Webhook 回调使用指南

本 Notebook 演示如何通过 **七牛 AIToken** 平台使用 **Webhook 回调** 模式调用模型 API。

## Webhook 与轮询的区别

在之前的示例中，我们使用「轮询」模式：提交请求后不断查询状态，直到任务完成。Webhook 提供了另一种方式：

| 模式 | 工作方式 | 适用场景 |
|------|---------|----------|
| **轮询（Polling）** | 客户端定时调用 `status()` 查询任务状态 | 需要实时展示进度的交互场景 |
| **Webhook 回调** | 任务完成后，服务端主动 POST 结果到你指定的 URL | 长时间任务、后台批处理、无需保持连接的场景 |

Webhook 的优势：
- **无需保持连接**：提交请求后即可释放客户端资源
- **适合长任务**：视频生成等耗时较长的任务不需要持续轮询
- **可靠重试**：webhook 投递失败会自动重试最多 10 次（2 小时内）

本示例使用 **Vidu Q3 Turbo Image-to-Video** 模型（`fal-ai/vidu/q3/image-to-video/turbo`）作为演示。

## 1. 环境准备

安装依赖包并配置环境变量。

In [ ]:
# 安装 fal-client
%pip install fal-client -q

In [ ]:
import os

# 设置七牛 AIToken 队列服务地址
os.environ["FAL_QUEUE_RUN_HOST"] = "api.qnaigc.com/queue"

# 设置 FAL_KEY 环境变量
# 你也可以在终端中通过 export FAL_KEY="xxx" 来设置
api_key = os.environ.get("QINIU_API_KEY")
if not api_key:
    raise ValueError("环境变量 QINIU_API_KEY 未设置")
os.environ["FAL_KEY"] = api_key

In [ ]:
import fal_client

## 2. 使用 Webhook 提交请求

与轮询模式不同，使用 Webhook 时只需在 `submit` 时传入 `webhook_url` 参数。任务完成后，fal 会向该 URL 发送 POST 请求，附带完整的结果。

> **注意**：`webhook_url` 必须是一个可从公网访问的 HTTPS 地址。本地开发时可以使用 [ngrok](https://ngrok.com/) 等内网穿透工具暴露本地端口。

In [ ]:
# 将此 URL 替换为你自己的 webhook 接收地址
WEBHOOK_URL = "https://your-domain.com/api/fal/webhook"

# 提交请求并指定 webhook 回调地址
handler = fal_client.submit(
    "fal-ai/vidu/q3/image-to-video/turbo",
    arguments={
        "prompt": "A woman walking through a vibrant city street at night, neon lights reflecting off wet pavement",
        "image_url": "https://aitoken-public.qnaigc.com/example/generate-video/running-man.jpg",
    },
    webhook_url=WEBHOOK_URL,
)

# 提交后立即返回 request_id，无需等待任务完成
print(f"请求已提交，request_id: {handler.request_id}")
print(f"Webhook 回调地址: {WEBHOOK_URL}")
print("任务完成后结果将 POST 到上述地址，无需轮询")

## 3. Webhook 回调数据格式

任务提交后，fal 会向你的 `webhook_url` 发送多次 POST 请求，通知任务状态变化。

### 进行中回调

任务开始处理时，会收到 `IN_PROGRESS` 状态的回调：

```json
{
  "request_id": "qvideo-root-1775099137785117576",
  "gateway_request_id": "qvideo-root-1775099137785117576",
  "status": "IN_PROGRESS",
  "payload": null
}
```

### 成功回调

任务完成时，`status` 为 `OK`，`payload` 中包含生成结果：

```json
{
  "request_id": "qvideo-root-1775099137785117576",
  "gateway_request_id": "qvideo-root-1775099137785117576",
  "status": "OK",
  "payload": {
    "video": {
      "url": "https://example.com/path/to/video.mp4",
      "content_type": "video/mp4"
    }
  }
}
```

### 失败回调

任务失败时，`status` 为 `ERROR`，`payload` 中包含错误详情：

```json
{
  "request_id": "qvideo-root-1775099394322622863",
  "gateway_request_id": "qvideo-root-1775099394322622863",
  "status": "ERROR",
  "payload": {
    "detail": {
      "loc": ["body"],
      "msg": "未知错误",
      "type": "unknown_error",
      "url": ""
    }
  },
  "error": "未知错误"
}
```

### 重试策略

- 采用**指数退避重试**，最多重试 3 次，间隔分别为 2s、4s、8s
- **重要**：webhook 处理逻辑应设计为**幂等的**，同一 `request_id` 可能会收到重复投递

## 4. 更多资源

- [七牛 AIToken 文档](https://developer.qiniu.com/aitokenapi)
- [fal-client Python 文档](https://docs.fal.ai/reference/client-libraries/python/fal_client)
- [fal Webhook 文档](https://fal.ai/docs/model-apis/inference/webhooks)
- [七牛 API 调用示例文档](https://apidocs.qnaigc.com)